# OpenAI-Compatible Whisper Server for Subtitle Edit

This notebook sets up a high-performance, OpenAI-compatible Whisper server using `faster-whisper`. It is configured to run on Google Colab's GPU and is optimized for highly accurate **Indonesian** transcriptions. It supports files larger than 25MB by using chunked file streaming.

To use it, follow the steps chronologically below.

## Step 1: Install Server Dependencies
We install `faster-whisper` for high-speed GPU-optimized inference, `fastapi` and `uvicorn` to host the web server, and `pyngrok` to establish a public connection for Subtitle Edit.

In [28]:
!pip install faster-whisper fastapi uvicorn python-multipart pyngrok nest-asyncio -q

## Step 2: Establish the Public Tunnel (Ngrok)
Subtitle Edit on your computer needs an internet-accessible URL to reach this Colab server.
1. Register for a free account at [ngrok.com](https://ngrok.com/) to obtain your **Authtoken**.
2. Paste your authtoken in the code cell below and execute it to generate your public endpoint.

In [19]:
# Kill any existing background ngrok processes to resolve ERR_NGROK_6024
import os
print("Terminating lingering ngrok processes...")
os.system("killall ngrok 2>/dev/null")

from pyngrok import ngrok
import nest_asyncio

# Paste your ngrok authtoken below
NGROK_TOKEN = "3MDW465bK6plW6p7XZbZWpPvwWo_3oVpbYyQDEQ1qEJ2PV1e1"

if NGROK_TOKEN != "3MDW465bK6plW6p7XZbZWpPvwWo_3oVpbYyQDEQ1qEJ2PV1e1":
    ngrok.set_auth_token(NGROK_TOKEN)
    # Open a HTTP tunnel on port 8000
    public_url = ngrok.connect(8000).public_url
    print(f"* Ngrok tunnel successfully established!")
    print(f"* Provide this API URL to Subtitle Edit: {public_url}/v1")
else:
    print("Please configure a valid NGROK_TOKEN to generate your public URL.")

Terminating lingering ngrok processes...
Please configure a valid NGROK_TOKEN to generate your public URL.


In [21]:
from pyngrok import ngrok, exception
import nest_asyncio

# Paste your ngrok authtoken below
NGROK_TOKEN = "3MDW465bK6plW6p7XZbZWpPvwWo_3oVpbYyQDEQ1qEJ2PV1e1"

if NGROK_TOKEN and NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except exception.PyngrokNgrokError:
        # Ignore error if the authtoken is already saved in the config file
        pass

    try:
        # Open a HTTP tunnel on port 8000
        public_url = ngrok.connect(8000).public_url
        print(f"* Ngrok tunnel established!")
        print(f"* Provide this API URL to Subtitle Edit: {public_url}/v1")
    except Exception as e:
        print(f"Error establishing tunnel: {e}")
else:
    print("Please configure a valid NGROK_TOKEN to generate your public URL.")

* Ngrok tunnel established!
* Provide this API URL to Subtitle Edit: https://tinsmith-relatable-scrunch.ngrok-free.dev/v1


## Step 3: Launch the OpenAI-Compatible Whisper Server
Run the cell below to load the highly accurate `large-v3` model and start the API server. Keep this cell running while you are transcribing in Subtitle Edit.

In [23]:
import shutil
import os
import nest_asyncio
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from faster_whisper import WhisperModel
import uvicorn

# Patch the asyncio loop to allow running servers inside Jupyter/Colab notebooks
nest_asyncio.apply()

app = FastAPI()

# Enable CORS to allow Subtitle Edit streams without connection interruptions
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load a highly accurate model ('large-v3') optimized for Indonesian language transcription
print("Loading highly accurate faster-whisper model (large-v3)...")
whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Model loaded!")

# Root verification endpoints to support Subtitle Edit connection tests (GET, POST, and HEAD)
@app.get("/")
@app.get("/v1")
@app.post("/v1")
@app.head("/")
@app.head("/v1")
async def root_verification():
    return JSONResponse(content={"status": "running", "message": "OpenAI-Compatible Whisper Server is active"})

# Dual routes handling both singular and plural versions of the endpoint
@app.post("/v1/audio/transcriptions")
@app.post("/v1/audio/transcription")
async def transcribe_audio(
    file: UploadFile = File(...),
    model: str = Form("large-v3"),
    language: str = Form("id"),  # Defaulting language specifically to Indonesian ('id')
    response_format: str = Form("json")
):
    # Save uploaded file temporarily using chunked streaming to handle large files securely
    temp_file_path = f"temp_{file.filename}"
    try:
        with open(temp_file_path, "wb") as buffer:
            while True:
                chunk = await file.read(1024 * 1024) # 1MB chunk size
                if not chunk:
                    break
                buffer.write(chunk)

        # Transcribe file with the specified Indonesian parameters
        segments, info = whisper_model.transcribe(
            temp_file_path,
            language=language if language else "id",
            beam_size=5
        )

        text_output = " ".join([segment.text for segment in segments]).strip()

    finally:
        # Clean up the temporary file
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

    return JSONResponse(content={"text": text_output})

print("Starting server on port 8000...")
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio", timeout_keep_alive=600)
server = uvicorn.Server(config)
# Run the uvicorn server asynchronously under the current Colab event loop
await server.serve()

Loading highly accurate faster-whisper model (large-v3)...
Model loaded!
Starting server on port 8000...


INFO:     Started server process [2030]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [2030]


### Local Subtitle Transcription (SRT Format)
Run the cell below to transcribe your local file `/content/1027 1.wav` directly into standard SRT subtitle format using the loaded model.

In [ ]:
import os
import sys
import av

# 1. Install and configure NVIDIA cuBLAS libraries to solve the libcublas.so.12 missing error on GPU
try:
    import nvidia.cublas
    # Add the pip-installed NVIDIA libraries to LD_LIBRARY_PATH safely
    if hasattr(nvidia.cublas, "__file__") and nvidia.cublas.__file__ is not None:
        cublas_path = os.path.dirname(nvidia.cublas.__file__)
        lib_path = os.path.join(cublas_path, "lib")
        if os.path.exists(lib_path):
            os.environ["LD_LIBRARY_PATH"] = lib_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
except ImportError:
    print("Installing missing CUDA/cuBLAS runtime libraries...")
    os.system("pip install nvidia-cublas-cu12 -q")
    try:
        import nvidia.cublas
        if hasattr(nvidia.cublas, "__file__") and nvidia.cublas.__file__ is not None:
            cublas_path = os.path.dirname(nvidia.cublas.__file__)
            lib_path = os.path.join(cublas_path, "lib")
            if os.path.exists(lib_path):
                os.environ["LD_LIBRARY_PATH"] = lib_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    except Exception as e:
        print(f"Could not configure NVIDIA libraries: {e}")

# 2. Monkeypatch av.open safely to bypass the 'metadata_errors' parameter issue
original_av_open = av.open
def patched_av_open(*args, **kwargs):
    kwargs.pop('metadata_errors', None)
    return original_av_open(*args, **kwargs)
av.open = patched_av_open

def format_timestamp(seconds):
    """Converts seconds to SRT timestamp format (HH:MM:SS,mmm)"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int(round((seconds % 1) * 1000))
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

audio_path = "/content/1027 1.wav"

if os.path.exists(audio_path):
    print(f"Transcribing {audio_path}... Please wait.")

    from faster_whisper import WhisperModel
    model = None

    # Try to load/use the GPU model first
    try:
        if 'whisper_model' in globals():
            model = whisper_model
        else:
            print("Initializing new GPU model...")
            model = WhisperModel("large-v3", device="cuda", compute_type="float16")

        # Dry run to verify CUDA libs are successfully linked
        segments, info = model.transcribe(audio_path, language="id", beam_size=1)
        # Consume one segment to trigger execution
        next(iter(segments), None)
    except Exception as e:
        print(f"GPU transcription execution failed: {e}. Falling back to CPU mode...")
        model = WhisperModel("large-v3", device="cpu", compute_type="float32")

    # Final transcription run
    try:
        segments, info = model.transcribe(
            audio_path,
            language="id",
            beam_size=5
        )

        srt_output = []
        for idx, segment in enumerate(segments, 1):
            start_time = format_timestamp(segment.start)
            end_time = format_timestamp(segment.end)
            text = segment.text.strip()
            srt_output.append(f"{idx}\n{start_time} --> {end_time}\n{text}\n")

        # Save the SRT subtitle file
        output_srt_path = audio_path.rsplit('.', 1)[0] + ".srt"
        with open(output_srt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(srt_output))

        print(f"\n✓ Transcription complete! Saved to: {output_srt_path}\n")
        print("--- First 10 Subtitle Lines ---")
        print("\n".join(srt_output[:10]))

    except Exception as e:
        print(f"Transcription execution failed: {e}")
else:
    print(f"Audio file not found at '{audio_path}'. Please make sure you have uploaded the file into your Colab file explorer with the exact name.")

Transcribing /content/1027 1.wav... Please wait.
Initializing new GPU model...
GPU transcription execution failed: maximum recursion depth exceeded. Falling back to CPU mode...


### Note on Subtitle Edit Settings
In Subtitle Edit:
1. Select **Video** -> **Audio to text (Whisper)**.
2. Choose **Whisper API (OpenAI compatible)**.
3. Paste the generated Ngrok public URL (ending with `/v1`) in the API URL field.

In [ ]:
!git clone https://github.com/petewarden/openai-whisper-webapp

In [ ]:
from IPython.display import Audio
Audio("/content/openai-whisper-webapp/mary.mp3")

In [ ]:
from IPython.display import Audio
Audio("/content/openai-whisper-webapp/daisy_HAL_9000.mp3")

## Define the Transcribe Function

Now we've loaded the model, and have the code, this is the function that takes an audio file path as an input and returns the recognized text (and logs what it thinks the language is).

In [ ]:
def transcribe(audio):

    # load audio and pad/trim it to fit 30 seconds
    audio = whisper.load_audio(audio)
    audio = whisper.pad_or_trim(audio)

    # make log-Mel spectrogram and move to the same device as the model
    mel = whisper.log_mel_spectrogram(audio).to(model.device)

    # detect the spoken language
    _, probs = model.detect_language(mel)
    print(f"Detected language: {max(probs, key=probs.get)}")

    # decode the audio
    options = whisper.DecodingOptions()
    result = whisper.decode(model, mel, options)
    return result.text


## Test with Pre-Recorded Audio

Before we bring up the UI to allow you to record your own live audio, we're going to run the `transcribe()` function on a couple of MP3s we've downloaded. You should see `Mary had a little lamb, its fleece was white as snow, and everywhere that Mary went, the lamb was sure to go.` for `mary.mp3`, which I recorded as an example of clear audio. The second file is a lot harder to transcribe, with very distorted audio, but the model does a good job with `Tazy, Tazy, Tazy. Give me your answer to time after crazy all for the love of you. It won't be a stylish marriage`. You'll notice the transcript is cut off after 30 seconds, which is the default length for this notebook. It can be extended, but that's outside of the scope of this documentation.

In [ ]:
easy_text = transcribe("/content/openai-whisper-webapp/mary.mp3")
print(easy_text)

hard_text = transcribe("/content/openai-whisper-webapp/daisy_HAL_9000.mp3")
print(hard_text)

## Install the Web UI Toolkit

We'll be using gradio to provide the widgets we need to do audio recording.

In [ ]:
! pip install gradio -q

In [ ]:
import gradio as gr
import time

## Web Interface

After running this script, you should see two widgets below that you can use to record live audio and see the transcription, as described in the introduction.

In [ ]:

gr.Interface(
    title = 'OpenAI Whisper ASR Gradio Web UI',
    fn=transcribe,
    inputs=[
        gr.inputs.Audio(source="microphone", type="filepath")
    ],
    outputs=[
        "textbox"
    ],
    live=True).launch()

### 1. Install Server Dependencies
We need to install `faster-whisper` for fast inference, `fastapi` & `uvicorn` for the API server, `python-multipart` to handle file uploads from Subtitle Edit, and `pyngrok` to tunnel our local server to a public URL.

In [ ]:
!pip install faster-whisper fastapi uvicorn python-multipart pyngrok nest-asyncio -q

### 2. Set up Ngrok Tunnel
Subtitle Edit needs an internet-accessible URL to communicate with this Colab notebook.
1. Go to [ngrok.com](https://ngrok.com/) and create a free account to get an **Authtoken**.
2. Paste your authtoken in the variable below.

In [ ]:
from pyngrok import ngrok
import nest_asyncio

# Paste your ngrok authtoken here
NGROK_TOKEN = "3MDW465bK6plW6p7XZbZWpPvwWo_3oVpbYyQDEQ1qEJ2PV1e1"

if NGROK_TOKEN != "3MDW465bK6plW6p7XZbZWpPvwWo_3oVpbYyQDEQ1qEJ2PV1e1":
    ngrok.set_auth_token(NGROK_TOKEN)
    # Open a HTTP tunnel on port 8000
    public_url = ngrok.connect(8000).public_url
    print(f"* Ngrok tunnel established! Provide this URL to Subtitle Edit: {public_url}")
else:
    print("Please set your NGROK_TOKEN to establish a public connection.")

### 3. Run OpenAI-Compatible Whisper Server
This script runs a FastAPI app on port 8000 that exposes the `/v1/audio/transcriptions` endpoint required by Subtitle Edit.

In [ ]:
import shutil
import os
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from faster_whisper import WhisperModel
import uvicorn

app = FastAPI()

# Load a highly accurate model ('large-v3') instead of 'base' to support accurate Indonesian transcription
print("Loading highly accurate faster-whisper model (large-v3)...")
whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Model loaded!")

@app.post("/v1/audio/transcriptions")
async def transcribe_audio(
    file: UploadFile = File(...),
    model: str = Form("large-v3"),
    language: str = Form("id"),  # Defaulting language specifically to Indonesian ('id')
    response_format: str = Form("json")
):
    # Save uploaded file temporarily using chunked streaming to handle files larger than 25MB safely
    temp_file_path = f"temp_{file.filename}"
    try:
        with open(temp_file_path, "wb") as buffer:
            # Read file in 1MB chunks to keep memory usage low even for huge audio files
            while True:
                chunk = await file.read(1024 * 1024)
                if not chunk:
                    break
                buffer.write(chunk)

        # Transcribe file with the specified language parameter
        segments, info = whisper_model.transcribe(
            temp_file_path,
            language=language if language else "id",
            beam_size=5
        )

        text_output = " ".join([segment.text for segment in segments]).strip()

    finally:
        # Clean up the temporary file immediately after processing
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

    # Subtitle Edit expects a JSON response containing the text
    return JSONResponse(content={"text": text_output})

# Allow FastAPI to run inside notebook event loop
nest_asyncio.apply()

print("Starting server on port 8000...")
uvicorn.run(app, host="0.0.0.0", port=8000)